# Exploração de Dados Públicos Brasileiros

Este notebook realiza a análise exploratória dos dados coletados via pipeline de dados públicos.

## Fontes de Dados
- **IBGE**: Localidades (regiões, estados e municípios) e agregados SIDRA


In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

# Configuração de caminhos
DATA_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')

print('Bibliotecas carregadas com sucesso!')

## 1. Exploração de Dados IBGE

In [ ]:
# Carregar dados de estados (processed primeiro, raw como fallback)
estados_parquet = PROCESSED_DIR / 'estados'
estados_path = DATA_DIR / 'ibge' / 'estados.json'
if estados_parquet.exists():
    df_estados = pd.read_parquet(estados_parquet)
    print(f'Total de estados (processed): {len(df_estados)}')
elif estados_path.exists():
    df_estados = pd.read_json(estados_path)
    print(f'Total de estados (raw): {len(df_estados)}')
else:
    raise FileNotFoundError('Execute o coletor (src.collectors.ibge_collector) ou o transformer primeiro.')
df_estados.head(10)

In [ ]:
# Carregar dados de municípios (processed primeiro, raw como fallback).
# O JSON raw é aninhado (microrregiao.mesorregiao.UF...), então sem
# json_normalize as colunas 'sigla_uf'/'regiao_nome' não existem.
municipios_parquet = PROCESSED_DIR / 'municipios'
municipios_path = DATA_DIR / 'ibge' / 'municipios.json'
if municipios_parquet.exists():
    df_municipios = pd.read_parquet(municipios_parquet)
    print(f'Total de municípios (processed): {len(df_municipios)}')
elif municipios_path.exists():
    df_municipios = pd.json_normalize(pd.read_json(municipios_path).to_dict(orient='records'))
    df_municipios = df_municipios.rename(columns={
        'microrregiao.mesorregiao.UF.sigla': 'sigla_uf',
        'microrregiao.mesorregiao.UF.regiao.nome': 'regiao_nome',
    })
    print(f'Total de municípios (raw normalizado): {len(df_municipios)}')
else:
    raise FileNotFoundError('Execute o coletor (src.collectors.ibge_collector) ou o transformer primeiro.')
print(f'\nColunas disponíveis: {list(df_municipios.columns)}')
df_municipios.head()

In [ ]:
# Distribuição de municípios por estado
col_uf = 'sigla_uf' if 'sigla_uf' in df_municipios.columns else 'sigla'
assert col_uf in df_municipios.columns, f"Coluna de UF ausente. Colunas: {list(df_municipios.columns)}"
contagem_uf = df_municipios[col_uf].value_counts().head(10)
        
fig, ax = plt.subplots(figsize=(12, 6))
contagem_uf.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Top 10 Estados por Número de Municípios', fontsize=14)
ax.set_xlabel('UF')
ax.set_ylabel('Quantidade de Municípios')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
        
print(f'\nEstatísticas por estado:')
print(contagem_uf.describe())

In [ ]:
# Análise de municípios por região
col_regiao = 'regiao_nome' if 'regiao_nome' in df_municipios.columns else 'regiao-nome'
assert col_regiao in df_municipios.columns, f"Coluna de região ausente. Colunas: {list(df_municipios.columns)}"
contagem_regiao = df_municipios[col_regiao].value_counts()
    
fig, ax = plt.subplots(figsize=(10, 6))
contagem_regiao.plot(kind='pie', ax=ax, autopct='%1.1f%%', startangle=90)
ax.set_title('Distribuição de Municípios por Região', fontsize=14)
ax.set_ylabel('')
plt.tight_layout()
plt.show()

## 2. Resumo dos Dados Coletados

In [ ]:
# Resumo geral
print('=' * 60)
print('RESUMO DOS DADOS PÚBLICOS COLETADOS')
print('=' * 60)

print(f'\n📊 IBGE:')
print(f'   Estados: {len(df_estados)}')
print(f'   Municípios: {len(df_municipios)}')
col_uf = 'sigla_uf' if 'sigla_uf' in df_municipios.columns else 'sigla'
assert col_uf in df_municipios.columns, f"Coluna de UF ausente. Colunas: {list(df_municipios.columns)}"
print(f'   Estados com municípios: {df_municipios[col_uf].nunique()}')

print(f'\n{'=' * 60}')